# CARA-FinSent results explorer
Loads every `*_summary_*.csv` and renders a side-by-side comparison.


## 1. Mount Drive & clone repo
Set `USE_DRIVE=True` to persist `data/`, `results/`, `figures/` across Colab runtime resets. 
Set `REPO_URL` to override the auto-detected git remote.


In [ ]:
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/CARA-FinSent"  #@param {type:"string"}
REPO_URL = ""  #@param {type:"string"}  # leave blank to auto-detect from this notebook's repo

import os, subprocess
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
    WORK_DIR = Path(DRIVE_DIR)
else:
    WORK_DIR = Path("/content")

REPO_DIR = WORK_DIR / "cara-finsent-experiments"
if not REPO_DIR.exists():
    if not REPO_URL:
        # Best-effort auto-detect: try the repo this notebook lives in (works when notebook is opened from GitHub).
        REPO_URL = os.environ.get("REPO_URL", "")
    if not REPO_URL:
        raise RuntimeError("Set REPO_URL above (e.g. https://github.com/<user>/cara-finsent-experiments.git)")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Working in:", os.getcwd())


## 2. Install dependencies


In [ ]:
!pip -q install -r requirements.txt


## 3. Aggregate latest summaries


In [ ]:
import pandas as pd
from pathlib import Path
rows = []
for path in sorted(Path("results").rglob("*_summary_*.csv")):
    try:
        df = pd.read_csv(path)
        df["_source_file"] = path.name
        rows.append(df)
    except Exception as e:
        print("Skip", path, e)
if not rows:
    raise SystemExit("No *_summary_*.csv found in results/. Run some experiments first.")
all_summary = pd.concat(rows, ignore_index=True, sort=False)
cols_first = [c for c in ["model","accuracy","macro_f1","weighted_f1","mcc","ece_10_bins","brier_score","ablation_flags","seed","_source_file"] if c in all_summary.columns]
all_summary = all_summary[cols_first + [c for c in all_summary.columns if c not in cols_first]]
all_summary.sort_values("macro_f1", ascending=False, inplace=True, na_position="last")
all_summary


## 4. Plot top models by macro-F1


In [ ]:
import matplotlib.pyplot as plt
top = all_summary.dropna(subset=["macro_f1"]).head(15)
plt.figure(figsize=(10, 5))
plt.barh(top["model"].astype(str)[::-1], top["macro_f1"][::-1])
plt.xlabel("macro_f1"); plt.title("Top 15 models by macro_f1"); plt.tight_layout(); plt.show()


## ⤓ Download results
Zip `results/` + `figures/` for sharing.


In [ ]:
import shutil, datetime
ts = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
archive = shutil.make_archive(f"cara_results_{ts}", "zip", root_dir=".", base_dir="results")
print("Created:", archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    pass
